# Viz 3 — Regression Charts

Renders the regression-tier charts: case-study trajectories, the headline
regression coefficient forest plot, and the descriptive
HCI × Production-quartile scatter.

| Chart | Page heading | Artifact |
|---|---|---|
| 10 | ECI vs Resource Dependence (Comparator Trajectories) | panel.parquet (small slice) |
| 12 | Headline Regression Coefficients | reg_coef_main.csv |
| 13 | ECI vs HCI by Production Quartile | hci_production_scatter.csv |


In [ ]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from _style import (base_layout, save, artifact_path, GRID)


## Chart 10 — Case-study trajectory comparators

Three case-study countries: Chile, Azerbaijan, Republic of Congo. Each is
plotted from the panel directly (small slice — three countries, 25 years).


In [ ]:
panel = pd.read_parquet(artifact_path('panel.parquet'))

cases = {'CHL': '#c23a3a', 'AZE': '#4a6fa5', 'COG': '#2e7d4a'}
code_to_name = dict(panel.groupby('Country Code')['Country Name'].first())

fig10 = go.Figure()
for cc, col in cases.items():
    sub = panel[panel['Country Code'] == cc].sort_values('Year')
    if len(sub) == 0:
        continue
    nm = code_to_name.get(cc, cc)
    fig10.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers', name=nm,
        line=dict(color=col, width=2.5),
        marker=dict(size=5, color=col),
        hovertemplate=f'<b>{nm}</b> · %{{x}}: ECI=%{{y:.3f}}<extra></extra>',
    ))
fig10.update_layout(
    **base_layout(height=500, margin=dict(l=70, r=40, t=20, b=60)),
    xaxis=dict(title='Year', gridcolor=GRID, dtick=5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID,
               zeroline=True, zerolinecolor='#ccc'),
    legend=dict(x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig10, '10_eci_trajectory_comparators.html')


## Chart 12 — Headline regression coefficients

Forest plot of Model 3b's coefficients with 95% confidence intervals from
country-clustered standard errors. Lagged ECI is dropped from the chart
(but kept in the fit) so the rest of the regressors are visible.


In [ ]:
coef_main = pd.read_csv(artifact_path('reg_coef_main.csv'))

fig12 = go.Figure()
fig12.add_vline(x=0, line=dict(color='#ccc', width=1.5))
fig12.add_trace(go.Scatter(
    x=coef_main['coef'], y=coef_main['short'],
    mode='markers', name='Headline (with lagged ECI)',
    marker=dict(size=8, color='#c23a3a', symbol='circle'),
    error_x=dict(type='data',
                 array=coef_main['ci_hi'] - coef_main['coef'],
                 arrayminus=coef_main['coef'] - coef_main['ci_lo'],
                 color='#c23a3a', thickness=1.5),
    customdata=list(zip(coef_main['short'], coef_main['coef'],
                        coef_main['ci_lo'], coef_main['ci_hi'])),
    hovertemplate='<b>%{customdata[0]}</b><br>'
                  'Coef: %{customdata[1]:.4f}<br>'
                  '95%% CI: [%{customdata[2]:.4f}, %{customdata[3]:.4f}]'
                  '<extra></extra>',
))
fig12.update_layout(
    **base_layout(height=560, margin=dict(l=170, r=40, t=20, b=70)),
    xaxis=dict(title='Coefficient', gridcolor=GRID),
    yaxis=dict(gridcolor=GRID, tickfont=dict(size=11)),
    legend=dict(x=1.01, y=0.99, font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig12, '12_coef_comparison_3a_3b.html')


## Chart 13 — ECI vs HCI by production quartile (descriptive)

Scatter of ECI against log(HCI), one trace per per-capita production
quartile, with an OLS trendline fitted within each quartile. Demoted from
the headline section because the interaction terms in chart 12 are not
significant.


In [ ]:
sc = pd.read_csv(artifact_path('hci_production_scatter.csv'))

Q_COLORS = {'Q1 (lowest)': '#4a6fa5', 'Q2': '#2e7d4a',
            'Q3': '#c9a227', 'Q4 (highest)': '#c23a3a'}

fig13 = go.Figure()
for q in ['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)']:
    sub = sc[sc['Prod_Q'] == q]
    if len(sub) == 0:
        continue
    fig13.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub['Economic Complexity Index'],
        mode='markers', name=q,
        marker=dict(size=5, color=Q_COLORS[q], opacity=0.5),
        hovertemplate=f'{q}<br>HCI: %{{x:.2f}}<br>'
                      'ECI: %{y:.3f}<extra></extra>',
    ))
    if sub['log_HCI'].notna().sum() > 5:
        slope, intercept = np.polyfit(sub['log_HCI'],
                                       sub['Economic Complexity Index'], 1)
        xr = np.linspace(sub['log_HCI'].min(), sub['log_HCI'].max(), 50)
        fig13.add_trace(go.Scatter(
            x=xr, y=intercept + slope * xr, mode='lines',
            line=dict(color=Q_COLORS[q], width=2, dash='dot'),
            showlegend=False, hoverinfo='skip',
        ))
fig13.update_layout(
    **base_layout(height=520, margin=dict(l=70, r=40, t=20, b=60)),
    xaxis=dict(title='Human Capital Index (log)', gridcolor=GRID),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID,
               zeroline=True, zerolinecolor='#ccc'),
    legend=dict(title='NR Production quartile', x=1.01, y=0.99,
                font=dict(size=11),
                bgcolor='rgba(250,250,250,0.9)',
                bordercolor=GRID, borderwidth=1),
)
save(fig13, '13_eci_hci_production_interaction.html')


## Done

Regression charts written to `outputs/`. To change which case-study
countries are plotted in chart 10, edit the `cases` dict at the top of
this notebook — no prep refit needed.
